# 02 - ResNet-18 Model Training

This notebook prepares and trains a pretrained ResNet-18 model
for woven fabric structure classification.

The three fabric classes are:
- Plain
- Satin
- Twill

A five-fold Stratified Group Cross-Validation strategy is used
for model evaluation.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from PIL import Image

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Identify the project directory
current_dir = Path.cwd()

if current_dir.name == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

# Define data paths
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

metadata_path = PROCESSED_DIR / "metadata.csv"
folds_path = PROCESSED_DIR / "folds.csv"

# Load saved CSV files
metadata_df = pd.read_csv(metadata_path)
folds_df = pd.read_csv(folds_path)

print("Metadata shape:", metadata_df.shape)
print("Fold assignments shape:", folds_df.shape)

print("\nFold distribution:")
print(folds_df["test_fold"].value_counts().sort_index())

Metadata shape: (130, 9)
Fold assignments shape: (130, 4)

Fold distribution:
test_fold
1    27
2    23
3    30
4    26
5    24
Name: count, dtype: int64


Verify the metadata and image locations

In [4]:
print("Metadata columns:")
print(metadata_df.columns.tolist())

print("\nFirst five metadata records:")
display(metadata_df.head())

print("\nFirst five fold records:")
display(folds_df.head())

Metadata columns:
['Image id', 'Specific Mass', 'Warp', 'Weft', 'Texture', 'image_filename', 'image_path', 'image_exists', 'group_id']

First five metadata records:


,Image id,Specific Mass,Warp,Weft,Texture,image_filename,image_path,image_exists,group_id
0,1,140,32,30,twill,W001.jpg,d:\MyFiles\Work File\Master's\Sem3\Project\Fab...,True,10
1,2,140,32,30,twill,W002.jpg,d:\MyFiles\Work File\Master's\Sem3\Project\Fab...,True,10
2,3,140,32,30,twill,W003.jpg,d:\MyFiles\Work File\Master's\Sem3\Project\Fab...,True,10
3,4,140,32,30,twill,W004.jpg,d:\MyFiles\Work File\Master's\Sem3\Project\Fab...,True,10
4,5,140,32,30,twill,W005.jpg,d:\MyFiles\Work File\Master's\Sem3\Project\Fab...,True,10



First five fold records:


,image_filename,Texture,group_id,test_fold
0,W001.jpg,twill,10,2
1,W002.jpg,twill,10,2
2,W003.jpg,twill,10,2
3,W004.jpg,twill,10,2
4,W005.jpg,twill,10,2


In [8]:
# Define the image directory inside the project
IMAGE_DIR = PROJECT_ROOT / "data" / "raw" / "images"

# Verify the folder and image files
print("Image folder:", IMAGE_DIR)
print("Image folder exists:", IMAGE_DIR.exists())
print("W001.jpg exists:", (IMAGE_DIR / "W001.jpg").exists())

# Count image files
image_files = list(IMAGE_DIR.glob("*.jpg"))

print("Total image files:", len(image_files))

Image folder: d:\MyFiles\Work File\Master's\Sem3\Project\FabricNet-XAI\masters-capstone-ml-project-268468L\data\raw\images
Image folder exists: True
W001.jpg exists: True
Total image files: 130


Create the fabric dataset class

In [9]:
class_to_idx = {
    "plain": 0,
    "satin": 1,
    "twill": 2
}

idx_to_class = {
    value: key for key, value in class_to_idx.items()
}


class FabricDataset(Dataset):

    def __init__(self, dataframe, image_dir, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image_filename = row["image_filename"]
        image_path = self.image_dir / image_filename

        image = Image.open(image_path).convert("RGB")

        label = class_to_idx[row["Texture"]]

        if self.transform:
            image = self.transform(image)

        return image, label


print("FabricDataset class created successfully!")

FabricDataset class created successfully!


Define image preprocessing

In [10]:
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

preprocess = transforms.Compose([
    transforms.CenterCrop(480),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

print("Image preprocessing defined successfully!")

Image preprocessing defined successfully!


Prepare Fold 1 training and testing data

In [11]:
# Add the test_fold information to the metadata
model_df = metadata_df.merge(
    folds_df[["image_filename", "test_fold"]],
    on="image_filename",
    how="inner",
    validate="one_to_one"
)

# Select Fold 1
current_fold = 1

train_df = model_df[
    model_df["test_fold"] != current_fold
].copy()

test_df = model_df[
    model_df["test_fold"] == current_fold
].copy()

print("Current fold:", current_fold)
print("Training images:", len(train_df))
print("Testing images:", len(test_df))

print("\nTraining class distribution:")
print(train_df["Texture"].value_counts())

print("\nTesting class distribution:")
print(test_df["Texture"].value_counts())

Current fold: 1
Training images: 103
Testing images: 27

Training class distribution:
Texture
plain    40
twill    32
satin    31
Name: count, dtype: int64

Testing class distribution:
Texture
plain    10
satin    10
twill     7
Name: count, dtype: int64


Create training and testing DataLoaders

In [12]:
BATCH_SIZE = 16

# Create training dataset
train_dataset = FabricDataset(
    dataframe=train_df,
    image_dir=IMAGE_DIR,
    transform=preprocess
)

# Create testing dataset
test_dataset = FabricDataset(
    dataframe=test_df,
    image_dir=IMAGE_DIR,
    transform=preprocess
)

# Create training DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

# Create testing DataLoader
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Training and testing DataLoaders created!")
print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))

Training and testing DataLoaders created!
Training batches: 7
Testing batches: 2


Verify one training batch

In [13]:
images, labels = next(iter(train_loader))

print("Images batch shape:", images.shape)
print("Labels batch shape:", labels.shape)

print("\nFabric classes in this batch:")
print([
    idx_to_class[label.item()]
    for label in labels
])

Images batch shape: torch.Size([16, 3, 224, 224])
Labels batch shape: torch.Size([16])

Fabric classes in this batch:
['twill', 'satin', 'plain', 'plain', 'twill', 'twill', 'satin', 'plain', 'twill', 'twill', 'twill', 'satin', 'plain', 'plain', 'twill', 'twill']


## Step 20 - Load Pretrained ResNet-18

Load an ImageNet-pretrained ResNet-18 model and adapt its
final classification layer for the three woven fabric classes.